# K2 · Evaluate a completed checkpoint, no retraining

Gắn **ZIP kết quả K2/baseline + Cityscapes + ACDC**, bật **GPU + Internet**, Run All. Dùng được cho recipe20epoch cũ và balanced_v2 đã đủ horizon. Nếu có nhiều ZIP, đặt RESULT_INPUT.

Kiểm seal/hash/source trước khi đọc target; giữ nguyên weights. Báo Cityscapes val500, ACDC val406 theo từng thời tiết/lớp, confusion và ảnh RGB/GT/prediction/error. Đây không phải hidden-test score hoặc K4 adaptation; không tự chọn checkpoint bằng ACDC.

In [ ]:
"""Standard-library source bootstrap shared by the two DG model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_dg_suite.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os, sys, subprocess
RESULT_INPUT = None  # Attach one full/shared K2 result ZIP/folder; set explicitly when multiple results exist.
CITYSCAPES_ROOT = None
ACDC_ROOT = None
ACDC_MASK_ROOT = None
CODE_INPUT = None
GIT_REF = 'feat/k1-weather-augmentation'
BASE = Path('/kaggle/working')
REPO = BASE/'k2_evaluation_code'
EVAL_ROOT = BASE/'k2_post_training_evaluation'
EVAL_ARCHIVE = BASE/'k2_post_training_evaluation.zip'
FIGURES = EVAL_ROOT/'figures'
assert not REPO.exists() and not EVAL_ROOT.exists(), 'Use a fresh session/output; preserve existing evaluation'
if CODE_INPUT is not None:
    copy_source(resolve_source(Path(CODE_INPUT),Path('/kaggle/input')),REPO)
else:
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',GIT_REF,
        'https://github.com/MinhToan1001/adverse-weather-segmentation.git',str(REPO)],check=True)
assert (REPO/'scripts/evaluate_k2.py').is_file(), 'Need the latest K2 evaluation code bundle'
os.chdir(REPO);sys.path.insert(0,str(REPO))
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
print('Evaluation only: no optimizer, no target weight updates, no training restart.')


In [ ]:
def run(*args):
    subprocess.run([sys.executable,'-u',*map(str,args)],check=True,cwd=REPO)
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow>=10,<12',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torch
import torchvision
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) >= (2,6), 'Need Torch>=2.6'
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


In [ ]:
from scripts.evaluate_k2 import resolve_result
from src.experiments.packaging import export_workspace
from IPython.display import FileLink, display
RESULT_INPUT = resolve_result(RESULT_INPUT,'/kaggle/input')
print('Reading frozen result:',RESULT_INPUT)
args = ['-m','scripts.evaluate_k2','--input',RESULT_INPUT,'--out',EVAL_ROOT,
        '--input-root','/kaggle/input','--device','cuda']
if CITYSCAPES_ROOT is not None: args += ['--city',str(CITYSCAPES_ROOT)]
if ACDC_ROOT is not None: args += ['--acdc',str(ACDC_ROOT)]
if ACDC_MASK_ROOT is not None: args += ['--acdc-masks',str(ACDC_MASK_ROOT)]
try:
    run(*args)
finally:
    try:
        if EVAL_ROOT.exists():
            run('-m','scripts.plot_k2','--input',RESULT_INPUT,'--out',FIGURES,'--evaluation',EVAL_ROOT)
    finally:
        if EVAL_ROOT.exists():
            export_workspace(EVAL_ROOT,EVAL_ARCHIVE)
            print('Evaluation ZIP:',EVAL_ARCHIVE);display(FileLink(str(EVAL_ARCHIVE)))
from IPython.display import Image as NotebookImage, display, FileLink
for name in ['training_pipeline','k2_architecture','learning_curves','class_learning','best_class_iou','condition_learning','weather_miou']:
    path = FIGURES / (name+'.png')
    if path.is_file(): display(NotebookImage(filename=str(path)))
if EVAL_ROOT.exists():
    for path in sorted(EVAL_ROOT.glob('*/panels/*.png')):
        display(NotebookImage(filename=str(path)))


Tải k2_post_training_evaluation.zip trong Output. Nếu lần đánh giá trước bị ngắt, giữ ZIP để kiểm tra rồi dùng phiên/output mới cho lần đánh giá lại cùng checkpoint. Nhãn test ACDC chính thức được giữ trên server; muốn điểm test cần nộp dự đoán riêng.